**Run evidence** · task `gold_material` · run `873128252550372` · result **SUCCESS** · executed 2026-10-19 17:27 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `02_build_gold/02a_material`.

# 02a — Workflow A → MATERIAL (grain: MATNR + WERKS)

Consolidates material master data into one row per material + plant.

**Steps (spec §4, Workflow A):**
1. Start from `RAW_MARC` (already at MATNR+WERKS grain).
2. Left-join `RAW_MARA` on MATNR, broadcasting MTART/MATKL to every plant row.
3. Filter `RAW_MAKT` to a single language (EN) **before** joining on MATNR, to
   avoid duplicating rows across languages.
4. Output: one row per MATNR+WERKS with MAKTX, MTART, MATKL, DISPO, SOBSL.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from pyspark.sql import functions as F

marc = spark.table(CFG.fq("RAW_MARC"))   # MATNR, WERKS, DISPO, SOBSL  (target grain)
mara = spark.table(CFG.fq("RAW_MARA"))   # MATNR, MTART, MATKL
makt = spark.table(CFG.fq("RAW_MAKT"))   # MATNR, SPRAS, MAKTX

# Step 3: filter descriptions to English FIRST so the join stays 1:1 on MATNR.
makt_en = makt.filter(F.col("SPRAS") == F.lit("EN")).select("MATNR", "MAKTX")

material = (
    marc
    .join(mara, on="MATNR", how="left")       # broadcast MTART/MATKL to every plant row
    .join(makt_en, on="MATNR", how="left")    # attach EN description
    .select(
        "MATNR", "WERKS", "MAKTX", "MTART", "MATKL", "DISPO", "SOBSL",
    )
)

(material.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(CFG.fq("MATERIAL")))

cnt = spark.table(CFG.fq("MATERIAL")).count()
# Grain assertion: MATNR+WERKS must be unique.
dupes = (spark.table(CFG.fq("MATERIAL"))
         .groupBy("MATNR", "WERKS").count().filter("count > 1").count())
print(f"MATERIAL rows: {cnt:,} | grain violations (MATNR+WERKS): {dupes}")
assert dupes == 0, "MATERIAL grain violated — MATNR+WERKS not unique!"
display(spark.table(CFG.fq("MATERIAL")).limit(10))

MATERIAL rows: 117 | grain violations (MATNR+WERKS): 0


MATNR,WERKS,MAKTX,MTART,MATKL,DISPO,SOBSL
MAT100000,1010,Component 000 assembly,FERT,CHEM,103,40
MAT100001,1010,Component 001 assembly,ROH,MECH,101,30
MAT100001,1030,Component 001 assembly,ROH,MECH,103,40
MAT100002,1010,Component 002 assembly,FERT,CHEM,103,
MAT100002,1020,Component 002 assembly,FERT,CHEM,101,40
MAT100003,1030,Component 003 assembly,ROH,MECH,102,
MAT100003,1020,Component 003 assembly,ROH,MECH,102,
MAT100004,1030,Component 004 assembly,ROH,ELEC,101,30
MAT100005,1030,Component 005 assembly,ROH,MECH,102,30
MAT100005,1010,Component 005 assembly,ROH,MECH,101,30


In [ ]:
dbutils.notebook.exit(f"02a_material: SUCCESS — {cnt} rows.")

Notebook exited: 02a_material: SUCCESS — 117 rows.
